In [ ]:
from pathlib import Path
import os
import re
import sys

required = {"inventory.csv", "inventory.py", "test_inventory.py"}
candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/Group_6_UGREEN_Submission")]
project = next((p for p in candidates if (p / "data/inventory.csv").exists() and all((p / "python" / name).exists() for name in required if name.endswith(".py"))), None)
if project is None:
    uploaded = {re.sub(r" \(\d+\)(?=\.[^.]+$)", "", Path(name).name): content for name, content in globals().get("uploaded", {}).items()}
    if required - uploaded.keys():
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError("Open this notebook from inside Group_6_UGREEN_Submission.")
        print("Select inventory.csv, inventory.py, and test_inventory.py together.")
        uploaded = files.upload()
        uploaded = {re.sub(r" \(\d+\)(?=\.[^.]+$)", "", Path(name).name): content for name, content in uploaded.items()}
    missing = required - uploaded.keys()
    if missing:
        raise FileNotFoundError("Missing files: " + ", ".join(sorted(missing)))
    project = Path("/content/Group_6_UGREEN_Submission")
    for name in required:
        target = project / ("data" if name.endswith(".csv") else "python") / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(uploaded[name])

os.chdir(project)
sys.path.insert(0, str(project / "python"))
print("Ready:", project)


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
from inventory import validate_data, score_inventory, rank_inventory, compare_methods

data = validate_data(pd.read_csv("data/inventory.csv"))
display(data.head())


In [ ]:
scored = score_inventory(data)
weighted = rank_inventory(scored, "weighted")
rule = rank_inventory(scored, "rule")
cols = ["rank", "item_id", "item_name", "weighted_score", "rule_score", "suggested_quantity", "order_cost_php", "funding_gap_units"]
display(weighted[cols].head(10))
display(rule[cols].head(10))


In [ ]:
comparison, metrics = compare_methods(scored)
display(pd.DataFrame([metrics]).T.rename(columns={0:"Value"}))
display(comparison.head(15))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(comparison.weighted_rank, comparison.rule_rank, color="#176a75")
ax.plot([1,len(data)],[1,len(data)],"--",color="gray")
ax.set(xlabel="Weighted rank (1 = highest)", ylabel="Rule-based rank (1 = highest)", title="Agreement between the two priority methods")
plt.show()
alternative=rank_inventory(score_inventory(data, {"shortage":.45,"urgency":.45,"lead_time":.10}))
sensitivity=comparison[["item_id","weighted_rank"]].merge(alternative[["item_id","rank"]].rename(columns={"rank":"alternative_rank"}),on="item_id")
sensitivity["absolute_change"]=abs(sensitivity.weighted_rank-sensitivity.alternative_rank)
display(sensitivity.sort_values("absolute_change",ascending=False).head(10))


In [ ]:
import unittest
import test_inventory
suite=unittest.defaultTestLoader.loadTestsFromModule(test_inventory)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "Fix test failures before continuing."


In [ ]:
example = pd.DataFrame([dict(item_id="MANUAL-001", item_name="UGREEN Nexode 240W USB-C Cable with Digital Display 1M", stock_level=5, daily_usage=2, lead_time_days=7, urgency=4, item_budget_php=3000, unit_cost_php=250, safety_days=7)])
display(rank_inventory(score_inventory(example))[["item_name","weighted_score","rule_priority","suggested_quantity","order_cost_php","decision","explanation"]])
uncertain = example.copy()
uncertain["lead_time_days"] = 14
display(score_inventory(uncertain)[["target_stock","desired_quantity","suggested_quantity","funding_gap_units"]])


In [ ]:
import subprocess
import sys

export = subprocess.run(
    [sys.executable, "python/inventory.py"],
    check=True, capture_output=True, text=True,
)
print(export.stdout)
print("Results saved to:", project / "results")
